# 1.Package Import

In [1]:
from bmovie_code import gen_table1_subj_info
from pynwb import NWBHDF5IO
from pathlib import Path
import numpy as np

from new_code import data_summary, load_data_into_pds, load_data_into_dict

import pickle

# 2. Global file location (need to specify your own path)

In [3]:
NWB_DIR="/Volumes/Extreme SSD/GT/PSYC8803/data/bmovie/nwb_files"
BIDS_DIR="/Volumes/Extreme SSD/GT/PSYC8803/data/bmovie/bids_files"

In [5]:
gen_table1_subj_info.main(
    nwb_input_dir=NWB_DIR,
    bids_datadir=BIDS_DIR
)

In [ ]:
# nwb_df, bids_df = data_summary.run_summary(NWB_DIR, BIDS_DIR, max_nwb_files=3, max_bids_files_each=5)

# nwb_df.head(), bids_df.head()

# 3. Load labels and data

In [ ]:
# labels
with open("../assets/annotations/short_faceannots.pkl", "rb") as f:
    face_annots = pickle.load(f)

type(face_annots)
# print(face_annots)

In [ ]:
nwb_path = load_data_into_pds.find_nwb_file_for_subject(NWB_DIR, "CS41")
print("Using:", nwb_path)

io = NWBHDF5IO(str(nwb_path), "r", load_namespaces=True)
nwb = io.read()

print("\n=== acquisition keys ===")
print(list(nwb.acquisition.keys()))

print("\n=== processing modules ===")
print(list(nwb.processing.keys()))

for mod_name, mod in nwb.processing.items():
    print(f"\n--- processing[{mod_name}] keys ---")
    try:
        print(list(mod.data_interfaces.keys()))
    except Exception as e:
        print("cannot list data_interfaces:", e)

print("\n=== stimulus keys ===")
print(list(getattr(nwb, "stimulus", {}).keys()))

print("\n=== intervals ===")
print(list(getattr(nwb, "intervals", {}).keys()))

io.close()

In [ ]:
from pynwb import NWBHDF5IO
io = NWBHDF5IO("/Volumes/Extreme SSD/GT/PSYC8803/data/bmovie/nwb_files/000623/sub-CS41/sub-CS41_ses-P41CSR1_behavior+ecephys.nwb", "r", load_namespaces=True)
nwb = io.read()

ece = nwb.processing["ecephys"]
print("LFP_macro electrical_series keys:", list(ece["LFP_macro"].electrical_series.keys()))
print("LFP_micro electrical_series keys:", list(ece["LFP_micro"].electrical_series.keys()))

beh = nwb.processing["behavior"]
print("EyeTracking spatial_series keys:", list(beh["EyeTracking"].spatial_series.keys()))
print("PupilTracking time_series keys:", list(beh["PupilTracking"].time_series.keys()))

io.close()

In [4]:
sub_nums = [41]

data = load_data_into_dict.load_multimodal_subjects(
    sub_nums=sub_nums,
    nwb_root=NWB_DIR,
    bids_root=BIDS_DIR,
    max_nwb_samples=20000,
    load_fmri=False
)

print(data[41].keys())
print(data[41]['meta'])
print(data[41]["lfp_macro"].shape, data[41]["eye_gaze"].shape)
print("n_units:", data[41]["meta"]["n_units"], "firing_rate:", data[41]["firing_rate"].shape)
print("band keys:", list(data[41]["lfp_bandpower"].keys()))

dict_keys(['spikes', 'firing_rate', 'lfp_macro', 'lfp_micro', 'lfp_bandpower', 'eye_gaze', 'pupil', 'movie_time', 'bold', 'meta'])
{'sub': 41, 'bids_sub': 'p41cs', 'nwb_sub': 'CS41', 'nwb_path': '/Volumes/Extreme SSD/GT/PSYC8803/data/bmovie/nwb_files/000623/sub-CS41/sub-CS41_ses-P41CSR1_behavior+ecephys.nwb', 'fs_macro': np.float64(1000.0), 'fs_micro': np.float64(1000.0), 'n_units': 7, 'fmri_error': None}
(11971, 96) (11971, 2)
n_units: 7 firing_rate: (11971, 7)
band keys: ['theta', 'alpha', 'beta', 'gamma', 'high_gamma']


## Tucker's PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
#updated for file opening
import pandas as pd
import numpy as np
from pynwb import NWBHDF5IO
#filename will need to be altered to work in this
filename = r'C:\Users\tucke\nwb_files\sub-CS41\sub-CS41_ses-P41CSR2_behavior+ecephys.nwb'
io = NWBHDF5IO(filename, 'r')
nwbfile = io.read()
nwbfile
#-----end update
lfp_interface = nwbfile.processing['ecephys'].data_interfaces['LFP_macro']
series_key = list(lfp_interface.electrical_series.keys())[0]
lfp_series = lfp_interface.electrical_series[series_key]
lfp_matrix = lfp_series.data[:60000, :]

scaler = StandardScaler()
lfp_scaled = scaler.fit_transform(lfp_matrix)

pca = PCA(n_components=3)
pca_results = pca.fit_transform(lfp_scaled)

print(f"Explained variance: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

plt.figure(figsize=(10, 6))
plt.plot(pca_results[:, 0], label='PC1 (Main Trend)')
plt.plot(pca_results[:, 1], label='PC2 (Second Trend)')
plt.title("Brain Activity Trends over 60 Seconds")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude (Arbitrary Units)")
plt.legend()
plt.show()
